# Horizon lab — mini-flash sampling clocks

Sweep detection / intensity / kill-ladder usefulness across trade-count, calendar, and volume clocks on HL ETH.

**SoT:** `HORIZON_RECOMMENDATION.md` · `out/EXP_REPORT.md` · `run_horizon_sweep.py`

No live orders. ClickHouse MCP banned.

In [ ]:
from pathlib import Path
import json
from IPython.display import display, Markdown, Image

LAB = Path('.').resolve()
if not (LAB / 'out' / 'summary.json').exists():
    LAB = Path(os.environ.get("ARES_MICROSTRUCTURE") or (Path.home() / "srv" / "ares-microstructure")) / 'research' / 'books' / 'cross_miniflash' / 'applications' / 'horizon_lab'
OUT = LAB / 'out'
FIG = OUT / 'figs'
summary = json.loads((OUT / 'summary.json').read_text())
print('days_ok', summary['n_days_ok'], summary['day_min'], '→', summary['day_max'])
print('rec_paper:', summary['rec_paper'][:120], '...')
display(Markdown((LAB / 'HORIZON_RECOMMENDATION.md').read_text()))

## Detection-clock scoreboard

In [ ]:
import pandas as pd

rows = []
for c in summary['clock_board']:
    d = c.get('delta_abs_mo5') or {}
    rows.append({
        'clock': c['clock'],
        'gated/day': round(c['gated_per_day'], 3),
        'med_bar_s': None if c['med_bar_s'] != c['med_bar_s'] else round(c['med_bar_s'], 1),
        'n_fire': c['n_fire'],
        'Δ|mo|': None if d.get('delta') is None else round(d['delta'], 2),
        'lo': None if d.get('lo') is None else round(d['lo'], 2),
        'hi': None if d.get('hi') is None else round(d['hi'], 2),
        'Vshare_fire': None if c.get('v_share_fire') != c.get('v_share_fire') else round(c['v_share_fire'], 2),
        'stable': c.get('sign_stable_early_late'),
        'sec/day': round(c['mean_sec_per_day'], 3),
        'verdict': c['verdict'],
    })
display(pd.DataFrame(rows))

## Policy intensity (event detection fixed)

In [ ]:
prows = []
for p in summary['policy_board']:
    d = p.get('delta_abs_mo5') or {}
    prows.append({
        'policy': p['policy'],
        'kind': p.get('intensity_kind'),
        'val': p.get('intensity_val'),
        'n_fire': p['n_fire'],
        'Δ|mo|': None if d.get('delta') is None else round(d['delta'], 2),
        'lo': None if d.get('lo') is None else round(d['lo'], 2),
        'hi': None if d.get('hi') is None else round(d['hi'], 2),
        'stable': p.get('sign_stable_early_late'),
        'verdict': p['verdict'],
    })
display(pd.DataFrame(prows))

## Figures

In [ ]:
for name in [
    'fig_gated_rate_by_clock.png',
    'fig_delta_mo_by_clock.png',
    'fig_policy_intensity.png',
    'fig_tradeN_walltime.png',
    'fig_compute_cost.png',
]:
    p = FIG / name
    if p.exists():
        display(Markdown(f'### `{name}`'))
        display(Image(filename=str(p)))

## Liquidity slice

In [ ]:
display(Markdown('```json\n' + json.dumps(summary.get('liquidity', {}), indent=2) + '\n```'))
display(Markdown(summary.get('liquidity_blurb', '')))

## Re-run sweep

```bash
python3 run_horizon_sweep.py --workers 8
```